In [ ]:
# pip install --quiet --upgrade pymongo gpt4all sentence_transformers


Note: you may need to restart the kernel to use updated packages.


In [1]:
MONGODB_URI = ("mongodb://localhost:8000/?directConnection=true")

In [1]:
import pandas as pd
import pymongo
from pymongo import MongoClient
from sentence_transformers import SentenceTransformer
import logging
import os
import json
import requests
import voyageai
from dotenv import load_dotenv

load_dotenv()

# Connect to your local MongoDB deployment
# client = MongoClient(MONGODB_URI)
mongodb_connection = os.getenv("MongoDB_Client")
client = MongoClient(mongodb_connection)

# Select the sample_airbnb.listingsAndReviews collection
## Will need to add Zendesk Later for more robust data pulling purposes
collection = client["zendesk_ticket"]["Zoho_Ticket"]

pipeline = [
    {
        '$project': {
            'email': 1, 
            'createdTime': 1, 
            'subject': 1, 
            'program': 1, 
            'content': 1, 
            'closedTime': 1
        }
    }, {
        '$match': {
            '$expr': {
                '$gte': [
                    {
                        '$toDate': '$createdTime' # Need to convert Date String to Date Object for comparison
                    }, {
                        '$dateSubtract': {
                            'startDate': '$$NOW', 
                            'unit': 'day', 
                            'amount': 10
                        }
                    }
                ]
            }
        }
    }
]

results = collection.aggregate(pipeline)
result_list = list(results)
emails = [item['content'] for item in result_list]
emails
# zoho_ticket = pd.DataFrame(result_list) # Convert into Pandas DataFrame



['Dear Stephanie, I hope this email finds you well. My name is Qingying Huang (Student ID: X319823 ). For my personal reason, I plan to transfer to Edmonds College in Washington to begin their Summer quarter. Could you please assist me with the following: 1. Due to that I’m not going to study at KSE anymore, please help me drop all my courses of Spring Quarter 2026 at UCSB. 2. I am going to study at Edmonds College this summer. And I have been admitted. Please refer to the attachment of acceptance letter. I would like to request that my active SEVIS record be released to Edmonds College. SEVIS School Code: SEA214F00298000. 3. I have finished filling the request form and waiting for my transfer-out request. Thank you for your time and assistance in ensuring a smooth transition for my studies. I look forward to your guidance. Best regards, Qingying Huang',
 'Dear Liliana, I hope this email finds you well. My name is Qingying Huang (Student ID: X319823 ). For my personal reason, I plan to

In [3]:
print(len(emails))

93


### Create Embedding Function with Voyage AI + Storing Embedding Results in Collection  

In [2]:
from pymongo.operations import SearchIndexModel
import time
from dotenv import load_dotenv
from datetime import datetime, timedelta, timezone
# Load the embedding model (https://huggingface.co/mixedbread-ai/mxbai-embed-large-v1)
load_dotenv(override=True)

model = "voyage-4" # Voyage AI
vo = voyageai.Client(
    api_key=os.getenv("VOYAGE_API_KEY")
)
# print(os.getenv("Voyage_API_KEY"))
# Define a function to generate embeddings
def get_embedding(data):
  embeddings = vo.embed(data, model = model).embeddings
  return embeddings[0]
# print(os.getenv("VOYAGE_API_KEY"))



# model = SentenceTransformer('mixedbread-ai/mxbai-embed-large-v1')
# model.save(model_path)
# model = SentenceTransformer(model_path)

# Define function to generate embeddings
# def get_embedding(text):
#     return model.encode(text).tolist()

# Calculate the cutoff date (10 days ago)
# cutoff_date = datetime.now(timezone.utc) - timedelta(days=10)

# # Filters for only documents with a content field and without an embedding field, within last 10 days
# filter = {'$and': [ 
#     { 'content': { '$exists': True, "$nin": [ None, "" ] } }, 
#     { 'embedding': { '$exists': False } },
#     { 'createdTime': { '$gte': cutoff_date.isoformat() } }
# ]} 

# Creates embeddings for all matching documents (no limit) - Purpose is to store embeddings in the collection 
updated_doc_count = 0
for document in collection.find(filter):
    text = document['content']
    embedding = get_embedding(text)
    collection.update_one({ '_id': document['_id'] }, { "$set": { 'embedding': embedding } }, upsert=True)
    updated_doc_count += 1
    print(f"Processed {updated_doc_count} documents...")

print("Documents updated: {}".format(updated_doc_count))


<!-- ### Initialize/Create MongoDB Vector Search Index Model -->

In [4]:
index_name = "vector_index"
search_index_model = SearchIndexModel(
  definition = {
    "fields": [
      {
        "type": "vector",
        "numDimensions": 1024,
        "path": "embedding",
        "similarity": "cosine"
      }
    ]
  },
  name = index_name,
  type = "vectorSearch"
)
collection.create_search_index(model=search_index_model)

print("Polling to check if the index is ready. This may take up to a minute.")
predicate=None
if predicate is None:
   predicate = lambda index: index.get("queryable") is True
while True:
   indices = list(collection.list_search_indexes(index_name))
   if len(indices) and predicate(indices[0]):
      break
   time.sleep(5)
print(index_name + " is ready for querying.")

Polling to check if the index is ready. This may take up to a minute.
vector_index is ready for querying.


### Create Query Results and execute MQL pipeline

In [15]:
def get_query_results(query):
  """Gets results from a vector search query."""
  
  collection = client["zendesk_ticket"]["Zoho_Ticket"]

  query_embedding = get_embedding(query)
  pipeline = [
      {
            "$vectorSearch": {
              "index": "vector_index",
              "queryVector": query_embedding,
              "path": "embedding",
              "exact": True,
              "limit": 5
            }
      }, 
        # {
        #     "$addFields": {
        #         "createdTimeDate": {"$toDate": "$createdTime"}
        #     }
        # },
        # {
        #     "$match": {
        #         "$expr": {
        #             "$gte": [
        #                 "$createdTimeDate",
        #                 {
        #                     "$dateSubtract": {
        #                         "startDate": "$$NOW",
        #                         "unit": "day",
        #                         "amount": 10
        #                     }
        #                 }
        #             ]
        #         }
        #     }
        # },
        {
            "$project": {
                "_id": 0,
                "email": 1,
                "createdTime": 1,
                "createdTimeDate": 1,
                "subject": 1,
                "program": 1,
                "content": 1,
                "closedTime": 1,
                "score": {"$meta": "vectorSearchScore"}
            }
        }
  ]
  results = list(collection.aggregate(pipeline))
  emails = [item['content'] for item in results]
  # print(len(emails))
  return emails
# Test the function with a sample query
import pprint
pprint.pprint(get_query_results("International"))
# pprint.pprint(len(get_query_results("All Emails")))

['Hi team, The latest International Programs (UIP & ICP) evaluation results '
 'are now available. You can access the full report here: View Evaluation '
 'Report @Paolo Gardinali will follow-up with some historical perspective. '
 'Thanks everyone, and as always, great work supporting these programs. '
 'Best,-- Jonathan Engeln Lambinet-Lacson Manager, Automation & AI Integration '
 'Help & FAQ\xa0|\xa0 |\xa0 |\xa0 |\xa0',
 '-- Jonathan Engeln Lambinet-Lacson Marketing Specialist Help & FAQ | | | |',
 '-- Jonathan Engeln Lambinet-Lacson Marketing Specialist Help & FAQ | | | |',
 '-- Jonathan Engeln Lambinet-Lacson Marketing Specialist Help & FAQ | | | |',
 'Hello, Thanks for your email. I am currently traveling and away from my desk '
 'from March 23 to April 3 with limited access to email. I will respond to '
 'your message upon my return. For immediate assistance, please contact '
 'international@professional.ucsb.edu Feel free to check our Help Center for '
 'information on our Int

### Implement LLM SandBox API for LLM Integration

In [31]:
load_dotenv(override=True)

LLM_API_ENDPOINT = os.getenv("LLMSANDBOX_API_ENDPOINT")
LLM_API_KEY = os.getenv("LLMSANDBOX_API_KEY")

headers = {
    "x-api-key": LLM_API_KEY,
    "Content-Type": "application/json"
}

def llm_integration_chatbot(questions: str):

    documents = get_query_results(questions) # Generate Vector Search Queries
    context = "\n\n".join(documents)

    prompt = f"""Use the following context to answer the question. 
        Context: {context}
        
        Question: {questions}
    """ 
    # Full Prompt
    
    body = {"message": 
        {"role": "user", 
         "content": [{"contentType": "text", "body": prompt.strip()}], 
         "model": "claude-v4.5-sonnet"}}
    try:
        r = requests.post(f"{LLM_API_ENDPOINT}/conversation", headers=headers, json=body)
        r.raise_for_status()
        data_res = r.json()
        print(data_res)
    except requests.exceptions.HTTPError as e:
        print(f"HTTP error: {e}")
        print(f"Response body: {r.text}")
        return f"API Error: {e}"
    except Exception as e:
        print(f"Other error: {e}")
        return f"Error: {e}"
    
    conversation_id = data_res.get('conversationId')
    if not conversation_id:
        return "No conversationId returned from API."

    url = f"{LLM_API_ENDPOINT}/conversation/{conversation_id}"
    max_retries = 10
    wait_seconds = 2

    # Fetching the result immediately after posting, but the LLM hasn't finished generating yet.
    for attempt in range(max_retries): # This is to handle Time Asychronous issue. 
        try:
            response = requests.get(url, headers={"x-api-key": LLM_API_KEY})
            # response.raise_for_status()
            data_result = response.json()
            # print(data_result)

            # More robust way to get the response
            message_keys = list(data_result['messageMap'].keys())
            # print(message_keys)

            if not message_keys:
                print("No messages found in conversation.")
                print(f"Attempt {attempt + 1}: Assistant response not ready, retrying in {wait_seconds}s...")
                time.sleep(wait_seconds)
            

            # Try to get the assistant's response (usually the last message)
            # If there are multiple messages, get the last one (most recent)
            # If there's only one message, that might be the user's message, so return a default
            else:
                return data_result['messageMap'][message_keys[-1]]['content'][0]['body']
        except requests.exceptions.HTTPError as e:
            print(f"HTTP error: {e}")
            print(f"Response body: {r.text}")
            return f"API Error: {e}"
        except Exception as e:
            print(f"Other error: {e}")
            return f"Error: {e}"


result = llm_integration_chatbot("Please give me a summary of the International Program related tickets for the last few days. Thank you")
print(result)


{'conversationId': '01KMW1DX09TRFV029S293R39AB', 'messageId': '01KMW1DXBZ9D9AWC90FM1YST8V'}
No messages found in conversation.
Attempt 1: Assistant response not ready, retrying in 2s...
No messages found in conversation.
Attempt 2: Assistant response not ready, retrying in 2s...
No messages found in conversation.
Attempt 3: Assistant response not ready, retrying in 2s...
# Summary of International Program Related Tickets

Based on the recent communications, here are the key International Program tickets and inquiries:

## 1. **Program Evaluation Results Announcement**
- Jonathan Engeln Lambinet-Lacson shared the latest UIP & ICP evaluation results
- Full report available for team review
- Paolo Gardinali to provide historical perspective

## 2. **2026-2027 Academic Year Nominations Open**
- Official announcement from Liliana Lau that student nominations are now being accepted
- Information sessions available upon request
- Programs highlighted: UIP (University Immersion Program) and IC

### Ignore Below, Testing Purposes

In [20]:
# LLM_API_ENDPOINT = "https://6bk1nzbyfe.execute-api.us-east-1.amazonaws.com/api"
# LLM_API_KEY = "9edWpO1cce40LO34wjTbA2qlR7ZqbnCA4DnFjD8u"
load_dotenv(override=True)
LLM_API_ENDPOINT = os.getenv("LLMSANDBOX_API_ENDPOINT")
LLM_API_KEY = os.getenv("LLMSANDBOX_API_KEY")

headers = {
    "x-api-key": LLM_API_KEY,
    "Content-Type": "application/json"
}

# Ask Questions and Create Conversations
prompt = f"""SYSTEM: Please Give me a short summary based on the given information. "USER": {get_query_results("Summarize all tickets for the last ten days")}."""

body = {"message": 
        {"role": "user", 
         "content": [{"contentType": "text", "body": prompt.strip()}], 
         "model": "claude-v4.5-sonnet"}}

try:
    r = requests.post(f"{LLM_API_ENDPOINT}/conversation", headers=headers, json=body)
    r.raise_for_status()
    data_res = r.json()
    print(data_res)
except requests.exceptions.HTTPError as e:
    print("HTTP error:", e)
    print("Response body:", r.text)
except Exception as e:
    print("Other error:", e)

{'conversationId': '01KMVX2W155XR5K46BB5MRKHE0', 'messageId': '01KMVX2WD1F2FB7Q9JH7W25N44'}


In [21]:
url = f"{LLM_API_ENDPOINT}/conversation/01KMQ98GSSVA4AKQ20Y6TG57YH"
response = requests.get(url, headers={"x-api-key": LLM_API_KEY})
data = response.json()
print(data)

{'id': '01KMQ98GSSVA4AKQ20Y6TG57YH', 'title': '', 'createTime': 1774602961.7215586, 'messageMap': {'system': {'role': 'system', 'content': [{'contentType': 'text', 'body': ''}], 'model': 'claude-v4.5-sonnet', 'children': ['01KMQ98H5QFC9NFP4KP2SCFT4E'], 'feedback': None, 'usedChunks': None, 'parent': None, 'thinkingLog': None}, '01KMQ98H5QFC9NFP4KP2SCFT4E': {'role': 'user', 'content': [{'contentType': 'text', 'body': 'Use the following pieces of context to answer the question at the end. \n        ["Dear Extension Office, Hope you\'re doing well. I am a GAPP student under opt. I’m writing to double-check the status of my SEVIS record. My last day of work was January 31st, so my unemployment period officially kicked off on February 1st. I just wanted to make sure everything looks correct on your end and that my unemployment days are being tracked accurately in the system. I want to stay well within that 90-day limit. Could you let me know if my record is up to date? Also, if there\'s any

In [22]:
url = f"{LLM_API_ENDPOINT}/conversation/{data_res['conversationId']}"
response = requests.get(url, headers={"x-api-key": LLM_API_KEY})
data = response.json()
print(data)

{'id': '01KMVX2W155XR5K46BB5MRKHE0', 'title': '', 'createTime': 1774757965.861935, 'messageMap': {'system': {'role': 'system', 'content': [{'contentType': 'text', 'body': ''}], 'model': 'claude-v4.5-sonnet', 'children': ['01KMVX2WD1F2FB7Q9JH7W25N44'], 'feedback': None, 'usedChunks': None, 'parent': None, 'thinkingLog': None}, '01KMVX2WD1F2FB7Q9JH7W25N44': {'role': 'user', 'content': [{'contentType': 'text', 'body': 'SYSTEM: Please Give me a short summary based on the given information. "USER": ["Dear Extension Office, Hope you\'re doing well. I am a GAPP student under opt. I’m writing to double-check the status of my SEVIS record. My last day of work was January 31st, so my unemployment period officially kicked off on February 1st. I just wanted to make sure everything looks correct on your end and that my unemployment days are being tracked accurately in the system. I want to stay well within that 90-day limit. Could you let me know if my record is up to date? Also, if there\'s anythi

In [23]:
data_index = list(data['messageMap'].keys())
data_index

['system', '01KMVX2WD1F2FB7Q9JH7W25N44', '01KMVX32QNJXJHKBHJF631HQG2']

In [24]:
data_index = list(data['messageMap'].keys())
# print(data_index)
data['messageMap']["01KMQ9SHDCAYVZ9W1TFKBH8YF8"]['content'][0]['body']

KeyError: '01KMQ9SHDCAYVZ9W1TFKBH8YF8'

## Create Embeddings for all Email Tickets in MongoDB

In [10]:
from pymongo import UpdateOne
filter = {'$and': [ 
    { 'content': { '$exists': True, "$nin": [ None, "" ] } }, 
    { 'embedding': { '$exists': False }}
]} 

total = collection.count_documents(filter)
print(f"Total documents to update: {total}")

Total documents to update: 11943


In [ ]:
new_total = list(collection.find(filter).limit(500))
new_total

In [ ]:
for document in collection.find(filter).limit(500).batch_size(100):
    print(document['content'])

In [11]:
def _process_batch(batch_docs, collection, vo, model,
                   updated, failed, total, start_time):
    """
    Processes one batch:
    1. Sends all content to Voyage AI in ONE API call
    2. Writes all embeddings to MongoDB in ONE bulk operation
    """
    contents = [doc['content'] for doc in batch_docs]
    ids = [doc['_id'] for doc in batch_docs]


    try:
        # ONE Voyage AI API call for the entire batch
        embeddings = vo.embed(contents, model=model).embeddings

        # Build bulk operations — ONE MongoDB round trip for entire batch
        operations = [
            UpdateOne(
                {'_id': doc_id},
                {'$set': {'embedding': embedding}},
                upsert=True
            )
            for doc_id, embedding in zip(ids, embeddings)
        ]
        collection.bulk_write(operations, ordered=False)

        updated += len(batch_docs)

        # Rich progress tracking
        elapsed = round(time.time() - start_time, 1)
        percent = round(updated / total * 100, 1)
        rate = round(updated / elapsed) if elapsed > 0 else 0
        remaining = round((total - updated) / rate) if rate > 0 else "?"

        print(
            f"Progress: {updated}/{total} ({percent}%) | "
            f"{rate} docs/sec | "
            f"~{remaining}s remaining | "
            f"Failed: {failed}"
        )

    except Exception as e:
        # ONE batch fails, not the whole job
        print(f"Batch failed ({len(batch_docs)} docs): {e}")
        failed += len(batch_docs)

    return updated, failed

In [12]:
from itertools import islice
updated = 0
failed = 0
batch_doc = []
start_time = time.time()
# email_content = [email for email in batch_doc]

updated_doc_count = 0

cursor = collection.find(filter)

while True:
    batch = list(islice(cursor, 100))
    # print(batch)
    if not batch:
        break

    print(f"Processing batch of {len(batch)} documents")
    for document in batch:
        batch_doc.append(document)

        if len(batch_doc) >= 100:
            updated, failed = _process_batch(
                    batch_doc, collection, vo, model,
                    updated, failed, total, start_time
                )
            batch_doc = []  # reset batch after the list reaches 10
            
    if batch_doc:
        updated, failed = _process_batch(
                    batch_doc, collection, vo, model,
                    updated, failed, total, start_time)
elapsed = round(time.time() - start_time, 1)
print("-" * 50)
print(f"Backfill complete in {elapsed}s")
print(f"Successfully embedded: {updated}")
print(f"Failed: {failed}")

#     text = document['content']
#     embedding = get_embedding(text)
#     collection.update_one({ '_id': document['_id'] }, { "$set": { 'embedding': embedding } }, upsert=True)
#     updated_doc_count += 1
#     print(f"Processed {updated_doc_count} documents...")

# print("Documents updated: {}".format(updated_doc_count))

Processing batch of 100 documents
Progress: 100/11943 (0.8%) | 53 docs/sec | ~223s remaining | Failed: 0
Processing batch of 100 documents
Progress: 200/11943 (1.7%) | 32 docs/sec | ~367s remaining | Failed: 0
Processing batch of 100 documents
Progress: 300/11943 (2.5%) | 43 docs/sec | ~271s remaining | Failed: 0
Processing batch of 100 documents
Progress: 400/11943 (3.3%) | 52 docs/sec | ~222s remaining | Failed: 0
Processing batch of 100 documents
Progress: 500/11943 (4.2%) | 40 docs/sec | ~286s remaining | Failed: 0
Processing batch of 100 documents
Progress: 600/11943 (5.0%) | 42 docs/sec | ~270s remaining | Failed: 0
Processing batch of 100 documents
Progress: 700/11943 (5.9%) | 42 docs/sec | ~268s remaining | Failed: 0
Processing batch of 100 documents
Progress: 800/11943 (6.7%) | 37 docs/sec | ~301s remaining | Failed: 0
Processing batch of 100 documents
Progress: 900/11943 (7.5%) | 33 docs/sec | ~335s remaining | Failed: 0
Processing batch of 100 documents
Progress: 1000/11943 

## This is to update vector search and incldue program and createdTime Filter

In [1]:
from pymongo import MongoClient
from dotenv import load_dotenv
import os
import time

load_dotenv()

client = MongoClient(os.getenv("MongoDB_Client"))
collection = client["zendesk_ticket"]["Zoho_Ticket"]

print("Updating vector search index to add filter fields...")

collection.update_search_index(
    "vector_index",
    {
        "fields": [
            # Keep your existing vector field
            {
                "type": "vector",
                "numDimensions": 1024,
                "path": "embedding",
                "similarity": "cosine"
            },
            # Add these filter fields
            {
                "type": "filter",
                "path": "createdTime"
            },
            {
                "type": "filter",
                "path": "program"
            }
        ]
    }
)

print("Polling until index is ready...")
while True:
    indices = list(collection.list_search_indexes("vector_index"))
    if indices and indices[0].get("queryable") is True:
        print("Index is ready!")
        break
    print("Waiting...")
    time.sleep(5)

client.close()

Updating vector search index to add filter fields...
Polling until index is ready...
Index is ready!
